### Imports

In [1]:
# Import necessary modules
from utils import *
import utils.for_empatica as empatica
import utils.for_climate as climate
import pluma.schema.outdoor as outdoor

# Pluma-python API
from modules import *

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
# where getting raw and processed data from
rawdata = "/mnt/raid/emotional_data_raquel"

__________

### Check data location

In [6]:
datadir = os.path.join(rawdata, "data")

# find participants
participants = [
    p for p in os.listdir(datadir)
    if p.startswith("OE")
]
print(participants)

['OE011', 'OE015', 'OE019', 'OE010', 'OE024', 'OE018', 'OE005', 'OE004', 'OE007', 'OE022', 'OE021', 'OE020', 'OE009', 'OE012', 'OE002', 'OE017', 'OE023']


In [3]:
participant = participants[0]
participant_path = os.path.join(datadir, participant)
print(participant_path)

/mnt/raid/emotional_data_raquel/data/OE011


In [4]:
sessions = [
    s for s in os.listdir(participant_path)
    if os.path.isdir(os.path.join(participant_path, s))
]
print(sessions)

['Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z']


In [5]:
session = sessions[0]
session_path = os.path.join(participant_path, session)
print(session_path)

/mnt/raid/emotional_data_raquel/data/OE011/Copenhagen_Nordhavn_sub-OE203011_2024-06-25T113254Z


_________

## Data quality

In [22]:
import os
import pandas as pd

base_dir = os.path.join(rawdata, "fulldata_mine")

good_participants = set()
bad_participants = set()

for sub in os.listdir(base_dir):
    sub_path = os.path.join(base_dir, sub)
    if not os.path.isdir(sub_path):
        continue

    participant_is_bad = False  # track per participant

    for ses in os.listdir(sub_path):
        ses_path = os.path.join(sub_path, ses)
        file_path = os.path.join(ses_path, "alldata.csv")

        if not os.path.exists(file_path):
            print(f"❌ MISSING FILE: {sub} - {ses}")
            participant_is_bad = True
            continue

        try:
            df = pd.read_csv(file_path)

            if df.empty:
                print(f"⚠️ EMPTY FILE:  {sub} - {ses}")
                participant_is_bad = True
            else:
                print(f"✅ OK:          {sub} - {ses} ({len(df)} rows)")

        except Exception as e:
            print(f"🔥 CORRUPT FILE: {sub} - {ses} | Error: {e}")
            participant_is_bad = True

    # after checking all sessions for this participant
    if participant_is_bad:
        bad_participants.add(sub)
    else:
        good_participants.add(sub)

✅ OK:          sub-OE020 - ses-Norrebro (1587 rows)
✅ OK:          sub-OE020 - ses-Nordhavn (1502 rows)
✅ OK:          sub-OE005 - ses-Nordhavn (1383 rows)
✅ OK:          sub-OE005 - ses-Hellerup (1499 rows)
✅ OK:          sub-OE018 - ses-Hellerup (1602 rows)
⚠️ EMPTY FILE:  sub-OE012 - ses-Norreport
✅ OK:          sub-OE021 - ses-Norrebro (1263 rows)
✅ OK:          sub-OE021 - ses-Hellerup (1601 rows)
✅ OK:          sub-OE015 - ses-Norreport (1465 rows)
✅ OK:          sub-OE002 - ses-Hellerup (1473 rows)
✅ OK:          sub-OE022 - ses-Norrebro (1484 rows)
✅ OK:          sub-OE022 - ses-Norreport (1545 rows)
✅ OK:          sub-OE022 - ses-Nordhavn (1432 rows)
✅ OK:          sub-OE009 - ses-Norrebro (1374 rows)
✅ OK:          sub-OE009 - ses-Nordhavn (1670 rows)
⚠️ EMPTY FILE:  sub-OE007 - ses-Hellerup
✅ OK:          sub-OE019 - ses-Hellerup (1796 rows)
⚠️ EMPTY FILE:  sub-OE017 - ses-Norreport
✅ OK:          sub-OE023 - ses-Norrebro (1520 rows)
✅ OK:          sub-OE023 - ses-Norreport 

___________

## Bad participants reasoning

In [31]:
results = []

for sub in bad_participants:
    sub_path = os.path.join(rawdata, "data", sub.replace("sub-", ""))  # careful with naming!

    if not os.path.exists(sub_path):
        print(f"❌ Missing participant folder: {sub}")
        continue

    for session_folder in os.listdir(sub_path):
        session_path = os.path.join(sub_path, session_folder)

        if not os.path.isdir(session_path):
            continue

        session_name = extract_session_name(session_folder)

        # ---- define paths ----
        empatica_data_path = os.path.join(
            rawdata, 'supp_mine', 'stress_csv', sub, f'ses-{session_name}', '_1hz', 'data_all_1Hz.csv'
        )

        geodata_data_path = os.path.join(
            rawdata, 'supp', 'geodata', 'log', sub, f'ses-{session_name}',
            f'{sub}_ses-{session_name}_geodata.xlsx'
        )

        gaze_data_path = os.path.join(
            rawdata, 'supp_mine', 'gaze', sub, f'ses-{session_name}', 'gaze.csv'
        )

        eeg_data_path = os.path.join(
            rawdata,
            'analysis-climate_physio_eeg_model_pipeline-outdoor_literature',
            sub, f'ses-{session_name}',
            'data', 'a01_psd_compute.m', 'eeg_power.xlsx'
        )

        # ---- check existence ----
        results.append({
            "participant": sub,
            "session": session_name,
            "empatica": os.path.exists(empatica_data_path),
            "geodata": os.path.exists(geodata_data_path),
            "gaze": os.path.exists(gaze_data_path),
            "eeg": os.path.exists(eeg_data_path),
        })

df_check = pd.DataFrame(results)

In [32]:
df_check

,participant,session,empatica,geodata,gaze,eeg
0,sub-OE012,Norreport,False,False,True,False
1,sub-OE007,Hellerup,False,False,True,False
2,sub-OE017,Norreport,False,False,True,False


Missing core streams (geodata + empatica + eeg)
➜ leads to empty merged dataset </p>

The current pipeline is:</p>

GEODATA → base timeline</p>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓</p>
EMPATICA → merged</p>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓</p>
GAZE → merged</p>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;↓</p>
EEG → merged</p>

Therefore, without geodata, NOTHING works

________

## Good participants

In [26]:
def check_streams(df):
    return {
        "gps": df["original_latitude"].notna().any() if "original_latitude" in df else False,

        "empatica_hr": df["E4_HR"].notna().any() if "E4_HR" in df else False,

        "eda": df["eda_raw"].notna().any() if "eda_raw" in df else False,

        "gaze": (
            df["GazeX"].notna().any() and df["GazeY"].notna().any()
            if "GazeX" in df and "GazeY" in df else False
        ),

        "eeg": (
            df["alpha"].notna().any()
            if "alpha" in df else False
        ),

        "climate": df["temp_atmospheric"].notna().any() if "temp_atmospheric" in df else False,
    }

In [27]:
results = []

for sub in good_participants:
    sub_path = os.path.join(base_dir, sub)

    for ses in os.listdir(sub_path):
        file_path = os.path.join(sub_path, ses, "alldata.csv")

        df = pd.read_csv(file_path)

        streams = check_streams(df)

        results.append({
            "participant": sub,
            "session": ses,
            **streams
        })

df_streams = pd.DataFrame(results)

In [29]:
df_streams

,participant,session,gps,empatica_hr,eda,gaze,eeg,climate
0,sub-OE019,ses-Hellerup,True,False,False,True,False,True
1,sub-OE002,ses-Hellerup,True,True,True,True,True,True
2,sub-OE020,ses-Norrebro,True,True,True,True,True,True
3,sub-OE020,ses-Nordhavn,True,True,True,True,True,True
4,sub-OE021,ses-Norrebro,True,True,True,True,True,True
5,sub-OE021,ses-Hellerup,True,True,True,True,True,True
6,sub-OE018,ses-Hellerup,True,True,True,True,True,True
7,sub-OE024,ses-Nordhavn,True,True,True,True,False,True
8,sub-OE004,ses-Norreport,True,True,True,True,True,True
9,sub-OE010,ses-Nordhavn,True,True,True,True,False,True


In [30]:
df_streams[
    (df_streams["empatica_hr"] == False) |
    (df_streams["gaze"] == False) |
    (df_streams["eeg"] == False)
]

,participant,session,gps,empatica_hr,eda,gaze,eeg,climate
0,sub-OE019,ses-Hellerup,True,False,False,True,False,True
7,sub-OE024,ses-Nordhavn,True,True,True,True,False,True
9,sub-OE010,ses-Nordhavn,True,True,True,True,False,True
13,sub-OE015,ses-Norreport,True,False,False,True,False,True
21,sub-OE023,ses-Nordhavn,True,True,True,True,False,True
